# PSI200 event-based dataset

PSI200 се записва на неравни интервали (~10 мин), останалите сензори - по-често.

Тук **не** разтягаме всичко до 1 час. Вместо това:

- **един ред = едно четене на PSI200** (това е target-ът)
- **всяка feature = средна стойност на сензора в интервала между предишното и текущото четене на PSI200**

За всяка feature пазим 4 колони: `_mean`, `_std`, `_n` (брой сурови проби), `_frozen` (мъртъв сензор).

`PSI80` е алтернативен target - записва се като колона, но **не** влиза в features.

**Ред на изпълнение:** конфигурация -> read_raw -> build -> run, после всичко останало.

In [ ]:
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 250)

ENGINE = create_engine(
    'mssql+pyodbc:///?odbc_connect='
    'DRIVER={ODBC Driver 17 for SQL Server};SERVER=10.20.2.10;'
    'DATABASE=pulse;UID=Pulse_RO;PWD=PD@T@r3@der'
)

# feature -> { MillName: TagID }
SQL_TAGS = {
    'Ore':        {'Mill06': '500',  'Mill07': '455',  'Mill08': '467'},
    'WaterMill':  {'Mill06': '1219', 'Mill07': '461',  'Mill08': '474'},
    'WaterZumpf': {'Mill06': '1220', 'Mill07': '462',  'Mill08': '475'},
    'Power':      {'Mill06': '502',  'Mill07': '460',  'Mill08': '471'},
    'ZumpfLevel': {'Mill06': '501',  'Mill07': '458',  'Mill08': '470'},
    'PressureHC': {'Mill06': '1217', 'Mill07': '459',  'Mill08': '472'},
    'DensityHC':  {'Mill06': '1216', 'Mill07': '457',  'Mill08': '469'},
    'FE':         {'Mill06': '1762', 'Mill07': '1762', 'Mill08': '1762'},
    'PulpHC':     {'Mill06': '2777', 'Mill07': '2837', 'Mill08': '1833'},
    'PumpRPM':    {'Mill06': '2796', 'Mill07': '2856', 'Mill08': '1800'},
    'MotorAmp':   {'Mill06': '2770', 'Mill07': '2830', 'Mill08': '1805'},
    'PSI80':      {'Mill06': '5329', 'Mill07': '5331', 'Mill08': '5332'},
    'PSI200':     {'Mill06': '5328', 'Mill07': '5330', 'Mill08': '5333'},
}

TABLE_NAMES = ['LoggerValues'] + [
    f'LoggerValues_Archive_{m}{y}'
    for y in (2026, 2025)
    for m in ('Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec')
    if not (y == 2026 and m in ('Oct', 'Nov', 'Dec'))
]

TARGET, ALT_TARGET = 'PSI200', 'PSI80'
FEATURES = [f for f in SQL_TAGS if f not in (TARGET, ALT_TARGET)]
print(f'{len(FEATURES)} features:', FEATURES)

In [ ]:
def read_raw(start=None):
    """Сурови данни: {feature: {mill: Series}}. Без resample, без ffill."""
    tag_map = {}
    for feat, mills in SQL_TAGS.items():
        for mill, tag in mills.items():
            tag_map.setdefault(tag, []).append((feat, mill))

    tags = ','.join(tag_map)
    frames = []
    for tbl in TABLE_NAMES:
        q = f'SELECT IndexTime, LoggerTagID, Value FROM {tbl} WHERE LoggerTagID IN ({tags})'
        if start:
            q += f' AND IndexTime >= {start!r}'
        d = pd.read_sql_query(q, ENGINE)
        print(f'{tbl:<32} {len(d):>8}')
        if not d.empty:
            frames.append(d)

    raw = pd.concat(frames, ignore_index=True)
    raw['LoggerTagID'] = raw['LoggerTagID'].astype(str)
    raw['IndexTime'] = pd.to_datetime(raw['IndexTime'])
    # dedup по (време, таг) - иначе при еднакъв IndexTime губим мелници
    raw = raw.drop_duplicates(['IndexTime', 'LoggerTagID'], keep='last').sort_values('IndexTime')

    out = {}
    for tag, targets in tag_map.items():
        s = raw.loc[raw['LoggerTagID'] == tag].set_index('IndexTime')['Value'].sort_index()
        s = s[~s.index.duplicated(keep='last')]
        for feat, mill in targets:
            out.setdefault(feat, {})[mill] = s
    return out

In [ ]:
def build(raw, mill, lag=0, max_gap=60):
    """1 ред на PSI200 четене; features = mean/std/n в интервала до предишното четене."""
    psi = raw[TARGET][mill].dropna()
    edges = (psi.index - pd.Timedelta(minutes=lag)).values
    n = len(psi) - 1

    out = pd.DataFrame(index=psi.index[1:])
    out.index.name = 'TimeStamp'
    out[TARGET] = psi.values[1:]
    out['dt_sec'] = np.diff(psi.index.values) / np.timedelta64(1, 's')

    # алтернативен target (не е feature)
    out[ALT_TARGET] = raw[ALT_TARGET][mill].dropna().reindex(
        out.index, method='nearest', tolerance='15min')

    for feat in FEATURES:
        s = raw.get(feat, {}).get(mill)
        if s is None or s.dropna().empty:
            out[f'{feat}_mean'] = np.nan
            out[f'{feat}_std'] = np.nan
            out[f'{feat}_n'] = 0
            out[f'{feat}_frozen'] = True
            continue

        s = s.dropna()
        # в кой прозорец попада всяка сурова стойност
        pos = np.searchsorted(edges, s.index.values, side='left') - 1
        ok = (pos >= 0) & (pos < n)
        g = pd.DataFrame({'pos': pos[ok], 'v': s.values[ok]}).groupby('pos')['v']
        st = pd.DataFrame({'mean': g.mean(), 'std': g.std(), 'n': g.count()}).reindex(range(n))

        out[f'{feat}_mean'] = st['mean'].values
        out[f'{feat}_std'] = st['std'].values
        out[f'{feat}_n'] = st['n'].fillna(0).astype(int).values
        # замръзнал: >1 проба, но всички еднакви
        out[f'{feat}_frozen'] = ((st['n'] > 1) & (st['std'].fillna(0) == 0)).values

    out['n_features_ok'] = out[[f'{f}_mean' for f in FEATURES]].notna().sum(axis=1)
    if max_gap:
        out = out[out['dt_sec'] <= max_gap * 60]
    return out

In [ ]:
def clean(df, min_coverage=0.5):
    """Запълва NaN в features. Връща (df, keep, dropped).

    - features с покритие < min_coverage се изхвърлят (няма смисъл да се запълват)
    - _std при n == 1 -> 0 (има проба, но няма вариация)
    - празен прозорец (n == 0) -> ffill, после bfill за началото
    """
    coverage = {f: (df[f'{f}_n'] > 0).mean() for f in FEATURES}
    keep = [f for f in FEATURES if coverage[f] >= min_coverage]
    dropped = [f for f in FEATURES if f not in keep]

    out = df.copy()
    for f in keep:
        n = out[f'{f}_n']
        out[f'{f}_std'] = out[f'{f}_std'].mask(n == 1, 0.0).ffill().bfill()
        out[f'{f}_mean'] = out[f'{f}_mean'].ffill().bfill()

    # колко features са били запълнени в реда (не са имали нито една проба)
    out['n_ffilled'] = sum((out[f'{f}_n'] == 0).astype(int) for f in keep)

    out = out.drop(columns=[f'{f}_{s}' for f in dropped for s in ('mean', 'std', 'n', 'frozen')],
                   errors='ignore')
    out = out.drop(columns=['n_features_ok'], errors='ignore')
    return out, keep, dropped

In [ ]:
raw = read_raw('2026-10-01')
df, keep, dropped = clean(build(raw, 'Mill06'))

print('Изхвърлени (покритие < 50%):', dropped)
print('Останали features:', len(keep), '->', keep)
print()

feat_cols = [f'{f}_{s}' for f in keep for s in ('mean', 'std', 'n', 'frozen')]
counts = df[feat_cols].notna().sum()
print('Редове:', len(df))
print('Non-null по feature колони (уникални):', counts.unique())
print('Всички равни и пълни:', counts.nunique() == 1 and counts.iloc[0] == len(df))
print()

# PSI80 е алтернативен target и НЕ се запълва. Ако го ползваш като target:
# df = df[df['PSI80'].notna()].copy()

FEATURES = keep   # за да работят клетките по-долу
df.info()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
import matplotlib.pyplot as plt


def corr_block(kind):
    cols = [TARGET] + [f'{f}_{kind}' for f in FEATURES]
    return cols, df[cols].corr(method='spearman')


def draw(ax, cols, corr, title):
    im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, rotation=90, fontsize=8)
    ax.set_yticks(range(len(cols)))
    ax.set_yticklabels(cols, fontsize=8)
    for i in range(len(cols)):
        for j in range(len(cols)):
            v = corr.iat[i, j]
            ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7,
                    color='white' if abs(v) > 0.6 else 'black')
    ax.set_title(title)
    return im


fig, axes = plt.subplots(2, 1, figsize=(12, 20))

for ax, kind in zip(axes, ('mean', 'std')):
    cols, corr = corr_block(kind)
    im = draw(ax, cols, corr, f'Spearman correlation - {kind}')

fig.colorbar(im, ax=axes, shrink=.6, label='Spearman r')
plt.tight_layout()
plt.show()

In [ ]:
# 1. Features / target + хронологичен train/test split
# само _mean частта - _std се изключва
DROP_FEATURES = ['ZumpfLevel', 'Power']
FEAT_COLS = [f'{f}_mean' for f in FEATURES if f not in DROP_FEATURES]

X = df[FEAT_COLS].astype(float)
y = df[TARGET].astype(float)

# филтри преди split
mask_target = y.between(15, 30)          # PSI200 извън [15, 30]
mask_ore = X['Ore_mean'] >= 120          # Ore < 120
mask = mask_target & mask_ore

print(f'Изхвърлени по target (PSI200 < 15 или > 30): {int((~mask_target).sum())} от {len(y)}')
print(f'Изхвърлени по Ore_mean < 120:               {int((~mask_ore).sum())} от {len(y)}')
print(f'Остават след филтрите:                      {int(mask.sum())} реда')
X, y = X[mask], y[mask]

# ВАЖНО: данните са времеви ред -> split по време, НЕ random.
# Иначе моделът "вижда" бъдещето и резултатите са фалшиво добри.
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print()
print(f'X: {X.shape}   ({len(FEAT_COLS)} features, само _mean)')
print(f'train: {len(X_train):>6} реда   {X_train.index.min()}  ..  {X_train.index.max()}')
print(f'test : {len(X_test):>6} реда   {X_test.index.min()}  ..  {X_test.index.max()}')
print()
print(f'target {TARGET}: mean={y.mean():.3f}  std={y.std():.3f}  '
      f'min={y.min():.3f}  max={y.max():.3f}')
print(f'NaN в X: {int(X.isna().sum().sum())}   NaN в y: {int(y.isna().sum())}')
print()
print('Features:', FEAT_COLS)

In [ ]:
# 2. Preprocessing pipeline
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

# XGBoost е дървовиден модел -> не се нуждае от StandardScaler.
# Оставяме само imputer, за да е явно и да работи и с NaN.
num_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
])

print(num_pipeline)

In [ ]:
# 3. Helper за оценка на модел
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(name, model):
    """Fit-ва модела и връща метриките за train и test."""
    model.fit(X_train, y_train)
    pred_tr = model.predict(X_train)
    pred_te = model.predict(X_test)

    row = {
        'model': name,
        'rmse_train': mean_squared_error(y_train, pred_tr) ** 0.5,
        'rmse_test': mean_squared_error(y_test, pred_te) ** 0.5,
        'mae_test': mean_absolute_error(y_test, pred_te),
        'r2_test': r2_score(y_test, pred_te),
    }
    print(f'{name:<20} RMSE train={row["rmse_train"]:8.3f}   '
          f'test={row["rmse_test"]:8.3f}   '
          f'MAE={row["mae_test"]:7.3f}   R2={row["r2_test"]:6.3f}')
    return row


print('Готово.')

In [ ]:
# 4. XGBoost - обучение
from sklearn.pipeline import make_pipeline
from xgboost import XGBRegressor

xgb_reg = make_pipeline(num_pipeline, XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    reg_lambda=1.0,
    objective='reg:squarederror',
    tree_method='hist',
    random_state=42,
    n_jobs=-1,
))

xgb_result = evaluate('XGBoost', xgb_reg)

In [ ]:
# 5. Cross-validation (XGBoost)
from sklearn.model_selection import KFold, TimeSeriesSplit, cross_val_score

# KFold - както в книгата, но разбърква времето -> оптимистична оценка
kf = -cross_val_score(xgb_reg, X, y, scoring='neg_root_mean_squared_error',
                      cv=KFold(5, shuffle=True, random_state=42), n_jobs=-1)
# TimeSeriesSplit - train винаги е ПРЕДИ validation -> честна оценка
ts = -cross_val_score(xgb_reg, X, y, scoring='neg_root_mean_squared_error',
                      cv=TimeSeriesSplit(n_splits=5), n_jobs=-1)

print(f'KFold        RMSE: {kf.mean():8.3f} ± {kf.std():5.3f}   {np.round(kf, 2)}')
print(f'TimeSeries   RMSE: {ts.mean():8.3f} ± {ts.std():5.3f}   {np.round(ts, 2)}')
print()
print('KFold е по-нисък, защото "вижда" бъдещи редове. Гледай TimeSeries реда.')

In [ ]:
# 6. Feature importance (XGBoost)
import matplotlib.pyplot as plt

xgb = xgb_reg.named_steps['xgbregressor']
imp = pd.Series(xgb.feature_importances_, index=FEAT_COLS).sort_values(ascending=False)

print('Feature importance:')
print(imp.round(4).to_string())
print()
print(f'Сума на top 5: {imp.head(5).sum():.3f}   '
      f'брой features с importance < 0.01: {(imp < 0.01).sum()}')

ax = imp[::-1].plot.barh(figsize=(9, 6))
ax.set_xlabel('importance')
ax.set_title('XGBoost - feature importance')
ax.grid(axis='x', alpha=.3)
plt.tight_layout()
plt.show()

In [ ]:
# 7. Predictions vs actual (test set, XGBoost)
pred_test = xgb_reg.predict(X_test)

fig, axes = plt.subplots(2, 1, figsize=(14, 9))

axes[0].plot(y_test.index, y_test.values, label='actual', lw=1)
axes[0].plot(y_test.index, pred_test, label='predicted', lw=1, alpha=.8)
axes[0].set_title(f'{TARGET} - test set (XGBoost)')
axes[0].set_ylabel(TARGET)
axes[0].legend()
axes[0].grid(alpha=.3)

axes[1].scatter(y_test, pred_test, s=8, alpha=.4)
lims = [min(y_test.min(), pred_test.min()), max(y_test.max(), pred_test.max())]
axes[1].plot(lims, lims, 'r--', lw=1, label='perfect prediction')
axes[1].set_xlabel('actual')
axes[1].set_ylabel('predicted')
axes[1].set_title('actual vs predicted')
axes[1].legend()
axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

In [ ]:
# 8. Residuals (test set, XGBoost)
pred_test = xgb_reg.predict(X_test)
resid = y_test - pred_test

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(resid, bins=50)
axes[0].set_title('Residuals (actual - predicted)')
axes[0].set_xlabel('error')
axes[0].grid(alpha=.3)

axes[1].scatter(pred_test, resid, s=8, alpha=.4)
axes[1].axhline(0, color='r', ls='--', lw=1)
axes[1].set_xlabel('predicted')
axes[1].set_ylabel('residual')
axes[1].set_title('Residuals vs predicted')
axes[1].grid(alpha=.3)

plt.tight_layout()
plt.show()

print(resid.describe().round(3).to_string())
print()
print(f'Bias (mean residual): {resid.mean():.3f}')
print(f'RMSE: {(resid ** 2).mean() ** 0.5:.3f}')